# Entrenamiento ML

Objetivo: encontrar el modelo de clasificación supervisada que mejor detecte Objetos Potencialmente Peligrosos (PHA) usando las variables preprocesadas en la etapa anterior.

Se entrenan y comparan múltiples clasificadores bajo validación cruzada estratificada, usando F1-score de la clase PHA como criterio de selección. Se evalúan dos experimentos:

- **Modelo A** (`trabajo_preprocesado_moid.pickle`): incluye `moid`. Máximo poder predictivo.
- **Modelo B** (`trabajo_preprocesado.pickle`): excluye `moid`. Variables físicas puras, sin el criterio que define la etiqueta target.

## 1. Importar paquetes

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, precision_recall_curve, average_precision_score,
    classification_report
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

## 2. Carga de datos

Se cargan los dos datasets generados en la etapa de preprocesamiento. Ambos tienen las mismas transformaciones aplicadas (homogeneización, encoding, escalado, eliminación de variables redundantes); la única diferencia es la presencia o ausencia de `moid`.

In [2]:
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'README.md').exists():
    repo_root = repo_root.parent

if not (repo_root / 'README.md').exists():
    raise FileNotFoundError('No se encontro la raiz del repositorio (README.md).')

# Modelo A: con moid
ruta_moid = repo_root / '02_Datos' / '03_Trabajo' / 'trabajo_preprocesado_moid.pickle'
df_moid = pd.read_pickle(ruta_moid)

# Modelo B: sin moid
ruta_sin_moid = repo_root / '02_Datos' / '03_Trabajo' / 'trabajo_preprocesado.pickle'
df = pd.read_pickle(ruta_sin_moid)

print(f'Modelo A (con moid):  {df_moid.shape}')
print(f'Modelo B (sin moid):  {df.shape}')
print(f'\nColumnas Modelo A: {df_moid.columns.tolist()}')
print(f'\nColumnas Modelo B: {df.columns.tolist()}')

Modelo A (con moid):  (88292, 28)
Modelo B (sin moid):  (88292, 27)

Columnas Modelo A: ['pha', 'H', 'diameter', 'albedo', 'diameter_sigma', 'e', 'a', 'q', 'i', 'om', 'w', 'ma', 'tp', 'tp_cal', 'moid', 'sigma_e', 'sigma_a', 'sigma_q', 'sigma_i', 'sigma_om', 'sigma_w', 'sigma_ma', 'sigma_ad', 'sigma_n', 'sigma_tp', 'sigma_per', 'class', 'rms']

Columnas Modelo B: ['pha', 'H', 'diameter', 'albedo', 'diameter_sigma', 'e', 'a', 'q', 'i', 'om', 'w', 'ma', 'tp', 'tp_cal', 'sigma_e', 'sigma_a', 'sigma_q', 'sigma_i', 'sigma_om', 'sigma_w', 'sigma_ma', 'sigma_ad', 'sigma_n', 'sigma_tp', 'sigma_per', 'class', 'rms']


## 3. Experimento A — Modelo con `moid`

Se trabaja con `df_moid` (88.292 filas). `moid` se incluye como feature. Este experimento puede producir métricas infladas dado que `moid < 0.05 UA` forma parte de la definición formal de PHA — los resultados del Experimento B (sin `moid`) son el punto de referencia para evaluar el poder predictivo real de las variables físicas.

### 3.1 Separación de features y target

Se separa `pha` del resto de features. La separación se aplica sobre el dataset completo de trabajo — los folds de validación cruzada se generan a partir de aquí, sin dividir manualmente en train/test previo.

In [3]:
X_moid = df_moid.drop(columns=['pha'])
y_moid = df_moid['pha']

print(f'Features (X): {X_moid.shape}')
print(f'Target  (y): {y_moid.shape}')
print(f'\nDistribución de clases:')
print(y_moid.value_counts())
print(f'\nProporción PHA: {y_moid.mean():.4f} ({y_moid.sum()} positivos de {len(y_moid)} total)')

Features (X): (88292, 27)
Target  (y): (88292,)

Distribución de clases:
pha
0    88185
1      107
Name: count, dtype: int64

Proporción PHA: 0.0012 (107 positivos de 88292 total)


### 3.2 Validación cruzada estratificada

Se usa `StratifiedKFold` con 5 folds para garantizar que cada fold mantiene la proporción de PHA (~2%). Un `KFold` estándar podría generar folds sin ningún ejemplo positivo, produciendo métricas inútiles o errores en el cálculo de F1.

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Verificar que cada fold preserva la proporción de PHA
print(f'{"Fold":<6} {"Train PHA%":<12} {"Val PHA%":<12} {"Train n":<10} {"Val n":<10}')
print('-' * 50)
for fold, (train_idx, val_idx) in enumerate(cv.split(X_moid, y_moid), 1):
    train_pha = y_moid.iloc[train_idx].mean() * 100
    val_pha   = y_moid.iloc[val_idx].mean() * 100
    print(f'{fold:<6} {train_pha:<12.2f} {val_pha:<12.2f} {len(train_idx):<10} {len(val_idx):<10}')

Fold   Train PHA%   Val PHA%     Train n    Val n     
--------------------------------------------------
1      0.12         0.12         70633      17659     
2      0.12         0.12         70633      17659     
3      0.12         0.12         70634      17658     
4      0.12         0.12         70634      17658     
5      0.12         0.12         70634      17658     


### 3.3 Instanciación de modelos

Se instancian tres clasificadores con sus respectivas estrategias de balance de clases:

- **Logistic Regression** y **Random Forest**: `class_weight='balanced'` — sklearn ajusta internamente el peso de cada clase en proporción inversa a su frecuencia.
- **XGBoost**: `scale_pos_weight` — equivalente a `class_weight='balanced'`, calculado como la razón entre ejemplos negativos y positivos.

In [1]:
# Ratio para scale_pos_weight de XGBoost: negativos / positivos
neg = (y_moid == 0).sum()
pos = (y_moid == 1).sum()
spw = neg / pos
print(f"Negativos: {neg} | Positivos: {pos} | scale_pos_weight: {spw:.2f}")

logreg = LogisticRegression(
    class_weight="balanced",
    random_state=42,
    max_iter=1000
)

rfc = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

xgb = XGBClassifier(
    scale_pos_weight=spw,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

modelos = {"logreg": logreg, "rfc": rfc, "xgb": xgb}
print("Modelos instanciados:", list(modelos.keys()))

NameError: name 'y_moid' is not defined

### 3.4 Grillas de hiperparametros

Espacio de busqueda de los tres modelos. Las grillas de LogReg y RF se recorren exhaustivamente con GridSearchCV. La de XGBoost se muestrea con RandomizedSearchCV dado su mayor espacio combinatorio.

In [ ]:
# Logistic Regression -- 5 x 2 = 10 combinaciones x 5 folds = 50 fits
param_grid_logreg = {
    "C":       [0.01, 0.1, 1, 10, 100],
    "solver":  ["lbfgs", "saga"],
    "penalty": ["l2"]
}

# Random Forest -- 2 x 3 x 2 x 2 = 24 combinaciones x 5 folds = 120 fits
param_grid_rfc = {
    "n_estimators":      [100, 200],
    "max_depth":         [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf":  [1, 2]
}

# XGBoost -- espacio amplio, se muestrean 20 combinaciones x 5 folds = 100 fits
param_dist_xgb = {
    "n_estimators":     [100, 200, 300, 500],
    "max_depth":        [3, 4, 5, 6, 7],
    "learning_rate":    [0.01, 0.05, 0.1, 0.2],
    "subsample":        [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5]
}

print(f"LogReg  -- combinaciones: {5*2}")
print(f"RF      -- combinaciones: {2*3*2*2}")
print(f"XGBoost -- espacio total: {4*5*4*3*3*3} | muestreados: 20")

### 3.5 Instanciacion de busquedas

Criterio de optimizacion: `scoring="f1"` apunta a la clase positiva (PHA = 1). Cada buscador usa el mismo `cv` estratificado definido en 3.2.

In [ ]:
gs_logreg = GridSearchCV(
    estimator=logreg,
    param_grid=param_grid_logreg,
    scoring="f1",
    cv=cv,
    refit=True,
    n_jobs=-1,
    verbose=1
)
print("GridSearchCV -- Logistic Regression instanciado")

In [ ]:
gs_rfc = GridSearchCV(
    estimator=rfc,
    param_grid=param_grid_rfc,
    scoring="f1",
    cv=cv,
    refit=True,
    n_jobs=-1,
    verbose=1
)
print("GridSearchCV -- Random Forest instanciado")

In [ ]:
rs_xgb = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist_xgb,
    n_iter=20,
    scoring="f1",
    cv=cv,
    refit=True,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
print("RandomizedSearchCV -- XGBoost instanciado")

### 3.6 Entrenamiento y mejores configuraciones

Cada celda ejecuta el ajuste de forma independiente, registra el tiempo y muestra los mejores hiperparametros junto con el F1-score obtenido en validacion cruzada.

In [ ]:
t0 = time.time()
gs_logreg.fit(X_moid, y_moid)
t_logreg = time.time() - t0

print(f"Tiempo de entrenamiento : {t_logreg:.1f}s")
print(f"Mejores hiperparametros : {gs_logreg.best_params_}")
print(f"Mejor F1 (CV)           : {gs_logreg.best_score_:.4f}")

In [ ]:
t0 = time.time()
gs_rfc.fit(X_moid, y_moid)
t_rfc = time.time() - t0

print(f"Tiempo de entrenamiento : {t_rfc:.1f}s")
print(f"Mejores hiperparametros : {gs_rfc.best_params_}")
print(f"Mejor F1 (CV)           : {gs_rfc.best_score_:.4f}")

In [ ]:
t0 = time.time()
rs_xgb.fit(X_moid, y_moid)
t_xgb = time.time() - t0

print(f"Tiempo de entrenamiento : {t_xgb:.1f}s")
print(f"Mejores hiperparametros : {rs_xgb.best_params_}")
print(f"Mejor F1 (CV)           : {rs_xgb.best_score_:.4f}")